In [5]:
# =========================
# SETUP
# =========================
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from scipy import sparse

from src.models.train_svm import run_training
from src.utils.utils import load_csv

from src.config import (
    FINAL_TRAIN_SVM_PATH,
    FINAL_TEST_SVM_PATH,
    TRAIN_LABEL_PATH,
    TEST_PATH_STAGE2,
    SUBMISSION_SVM_PATH,
)

# LOAD TEST
X_test = sparse.load_npz(FINAL_TEST_SVM_PATH.with_suffix(".npz"))
test_df = load_csv(TEST_PATH_STAGE2)

# TRAIN 3 MODELS (AUTO TUNE C)
for model_type in ["svm", "logreg", "nb"]:
    print(f"\n====================")
    print(f"MODEL: {model_type.upper()}")
    print(f"====================")

    model_save_path = Path(f"../models/saved/{model_type}.pkl")

    model = run_training(
        train_path=FINAL_TRAIN_SVM_PATH,
        test_path=FINAL_TEST_SVM_PATH,
        label_path=TRAIN_LABEL_PATH,
        model_save_path=model_save_path,
        model_type=model_type,
        tune_C=True
    )

    # PREDICT
    y_test_pred = model.predict(X_test)

    submission = pd.DataFrame({
        "id": test_df["id"],
        "Label": y_test_pred
    })

    submission_path = SUBMISSION_SVM_PATH.with_name(
        f"{SUBMISSION_SVM_PATH.stem}_{model_type}{SUBMISSION_SVM_PATH.suffix}"
    )

    submission.to_csv(submission_path, index=False)
    print(f"✔ Saved submission → {submission_path}")

    print(f"✔ Saved submission → {submission_path}")


MODEL: SVM
Loading features...
Shape: (2496, 12508) (2496,)

Tuning C...

--- Testing C=0.1 ---

===== VALIDATION RESULTS =====
Accuracy: 0.326
F1-macro: 0.1849118336424485
              precision    recall  f1-score   support

           1       0.44      0.73      0.55       181
           2       0.17      0.05      0.08       103
           3       0.14      0.03      0.05        88
           4       0.11      0.01      0.02        74
           5       0.16      0.39      0.22        54

    accuracy                           0.33       500
   macro avg       0.20      0.24      0.18       500
weighted avg       0.25      0.33      0.25       500


--- Testing C=0.5 ---

===== VALIDATION RESULTS =====
Accuracy: 0.274
F1-macro: 0.23706186282715863
              precision    recall  f1-score   support

           1       0.50      0.42      0.46       181
           2       0.16      0.06      0.09       103
           3       0.17      0.27      0.21        88
           4       